# Regrid reanalyses to 1-degree grid

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import xagg as xa
import xesmf as xe
import os
import glob
import re

from funcs_support import get_filepaths,get_params,utility_save
dir_list = get_params()

In [2]:
from distributed import Client
client = Client()
display(client)

/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35851 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/35851/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/35851/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45889,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/35851/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:42841,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/34795/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:37733,


2026-02-24 13:43:17,514 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:42575 (pid=1757) exceeded 95% memory budget. Restarting...
2026-02-24 13:43:17,603 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:42575' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('sum-fe7a68b2018eab4a16ff4441128b43a3', 0, 1, 3, 0, 1), ('sum-partial-b12f641254f7444b139167acffa51aaa', 0, 0, 0, 1, 3), ('sum-fe7a68b2018eab4a16ff4441128b43a3', 0, 0, 2, 0, 1), ('sum-fe7a68b2018eab4a16ff4441128b43a3', 0, 0, 3, 0, 1), ('sum-fe7a68b2018eab4a16ff4441128b43a3', 0, 1, 0, 0, 0), ('sum-partial-b12f641254f7444b139167acffa51aaa', 0, 0, 1, 1, 1), ('sum-fe7a68b2018eab4a16ff4441128b43a3', 0, 0, 0, 0, 1), ('sum-partial-b12f641254f7444b139167acffa51aaa', 0, 0, 1, 1, 3), ('sum-fe7a68b2018eab4a16ff4441128b43a3', 0, 1, 2, 0, 1)} (stimulus_id='handle-worker-cleanup-1771965797.6027892')
2026-02-24 13:43:25,814 - distributed.nanny.memory - WARNING - Worker t

In [3]:
df = get_filepaths()

In [4]:
params_subset = {'lat':slice(-56,86),'lon':slice(-180,180)}
# Get reference 1x1 grid
ref_grid = xr.Dataset(coords = {'lat':(['lat'],np.arange(params_subset['lat'].start+0.5,
														 params_subset['lat'].stop-0.4)),
								'lon':(['lon'],np.arange(params_subset['lon'].start+0.5,
														 params_subset['lon'].stop-0.4))})


In [5]:
modlist = ['ERA5-025','MERRA2','JRA-3Q','GMFD']
varlist = ['tas','tasmax','pr','prcorr']

In [6]:
df = df.query('run == "reanalysis"')
df = df.iloc[[mod in modlist for mod in df.model.values],:]
df = df.iloc[[var in varlist for var in df.varname.values],:]

In [7]:
for row in df.iterrows():
    output_fn = re.sub(row[1]['model'],row[1]['model']+'-1deg',row[1]['path'])
    
    if not os.path.exists(output_fn):
        if row[1]['filetype'] == 'zarr':
            ds = xr.open_zarr(row[1]['path'])
        else:
            ds = xr.open_dataset(row[1]['path'])
        if 'pr' in row[1]['varname']:
            rgrd_typ = 'conservative'
        else:
            rgrd_typ = 'bilinear'
        rgrd = xe.Regridder(ds,ref_grid,rgrd_typ,ignore_degenerate=True)
        ds = rgrd(ds)
        
        ds.attrs['SOURCE'] = '0a_create_1deg_reanalyses.ipynb'
        ds.attrs['DESCRIPTION'] = f'Data, regridded using {rgrd_typ} algorithm to 1-degree BCD-ME grid from {re.split(r'\/',row[1]['path'])[-1]}'
        if row[1]['filetype'] == 'zarr':
            ds = ds.drop_encoding()
        utility_save(ds,output_fn)
    else:
        print(output_fn+' already processed, skipped.')

/glade/derecho/scratch/schwarzwald/climate_raw_derecho/MERRA2-1deg/prcorr_Amon_MERRA2-1deg_historical_reanalysis_19800101-20250530.nc already processed, skipped.
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/MERRA2-1deg/tas_day_MERRA2-1deg_historical_reanalysis_19800101-20240801.nc already processed, skipped.
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/MERRA2-1deg/tasmax_day_MERRA2-1deg_historical_reanalysis_19800101-20241231.nc already processed, skipped.
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/MERRA2-1deg/pr_day_MERRA2-1deg_historical_reanalysis_19800101-20231231.nc already processed, skipped.
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/MERRA2-1deg/pr_Amon_MERRA2-1deg_historical_reanalysis_19800101-20250530.nc already processed, skipped.
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/MERRA2-1deg/prcorr_day_MERRA2-1deg_historical_reanalysis_19800101-20241231.zarr already processed, skipped.
/glade/derecho/scratch/schwarzwald/climat

/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/climate_raw_derecho/ERA5-025-1deg/tasmax_day_ERA5-025-1deg_historical_reanalysis_19810101-20241231.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/climate_raw_derecho/ERA5-025-1deg/tas_day_ERA5-025-1deg_historical_reanalysis_19570101-20241231.zarr saved!


...

/glade/derecho/scratch/schwarzwald/climate_raw_derecho/GMFD-1deg/tas_day_GMFD-1deg_historical_reanalysis_19570101-20101231.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/computation/apply_ufunc.py:452: PerformanceWarning: Regridding is increasing the number of chunks by a factor of 8.0, you might want to specify sizes in `output_chunks` in the regridder call. Default behaviour is to preserve the chunk sizes from the input (100, 100).
  result_vars[name] = func(*variable_args)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
2026-02-24 13:43:16,913 - distributed.worker.memory - WARNING - gc.collect() took 1.065s. This is usually a sign that some tasks handle too many Python objects at the same time. Rechunking the work into smaller tasks might help.
2026-02-24 13:43:17,399 - distributed.worker.memory - WARNING -

/glade/derecho/scratch/schwarzwald/climate_raw_derecho/GMFD-1deg/tasmax_day_GMFD-1deg_historical_reanalysis_19810101-20081231.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/distributed/client.py:3375: UserWarning: Sending large graph of size 23.96 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/climate_raw_derecho/GMFD-1deg/pr_day_GMFD-1deg_historical_reanalysis_19570101-20101231.zarr saved!


In [8]:
ds

<xarray.Dataset> Size: 4GB
Dimensions:  (time: 19723, lat: 142, lon: 360)
Coordinates:
  * time     (time) datetime64[ns] 158kB 1957-01-01 1957-01-02 ... 2010-12-31
  * lat      (lat) float64 1kB -55.5 -54.5 -53.5 -52.5 ... 82.5 83.5 84.5 85.5
  * lon      (lon) float64 3kB -179.5 -178.5 -177.5 -176.5 ... 177.5 178.5 179.5
Data variables:
    pr       (time, lat, lon) float32 4GB dask.array<chunksize=(30, 142, 300), meta=np.ndarray>
Attributes:
    regrid_method:  conservative
    SOURCE:         0a_create_1deg_reanalyses.ipynb
    DESCRIPTION:    Data, regridded using conservative algorithm to 1-degree ...